<center>
    <img src="https://rockborne.com/wp-content/uploads/2021/07/LandingPage-Header-RED-CENTRE.jpg" width="900" alt="logo"  />
</center>

# PySpark Performance Optimisation
## Why Performance Matters

### The Business Context

Imagine you're a data engineer at a retail company processing daily sales data. Your pipeline currently takes **4 hours** to run. The business wants to refresh dashboards every hour. Without optimisation, this is impossible.

**Real costs of poor performance:**
- **Delayed insights:** Business decisions based on stale data
- **Higher cloud costs:** Longer cluster runtime = more money
- **Failed SLAs:** Reports delivered late, stakeholders frustrated
- **Blocked pipelines:** Downstream processes waiting for your job
- **Scalability issues:** Works on test data, fails on production volumes

### How Spark Works: A Quick Refresher

PySpark distributes data across a cluster as **partitions**. Each partition is processed independently by an executor. Understanding this is key to optimisation.

```
┌─────────────────────────────────────────────────────────────┐
│                        DRIVER                                │
│   (coordinates tasks, collects results)                      │
└─────────────────────────────────────────────────────────────┘
                              │
          ┌───────────────────┼───────────────────┐
          ▼                   ▼                   ▼
   ┌─────────────┐     ┌─────────────┐     ┌─────────────┐
   │  Executor 1 │     │  Executor 2 │     │  Executor 3 │
   │ ┌─────────┐ │     │ ┌─────────┐ │     │ ┌─────────┐ │
   │ │ Part 1  │ │     │ │ Part 3  │ │     │ │ Part 5  │ │
   │ └─────────┘ │     │ └─────────┘ │     │ └─────────┘ │
   │ ┌─────────┐ │     │ ┌─────────┐ │     │ ┌─────────┐ │
   │ │ Part 2  │ │     │ │ Part 4  │ │     │ │ Part 6  │ │
   │ └─────────┘ │     │ └─────────┘ │     │ └─────────┘ │
   └─────────────┘     └─────────────┘     └─────────────┘
```

**The Cluster Architecture:**
- **Driver:** The "brain" that plans and coordinates execution
- **Executors:** Workers that actually process data in parallel
- **Partitions:** Chunks of data that can be processed independently
- **Tasks:** Units of work, one task per partition

**The Three Sources of Performance Problems:**
1. **Data Movement (Shuffles)** - Moving data between executors across the network
2. **Data Skew** - Uneven partition sizes causing some tasks to take much longer
3. **Inefficient Operations** - Doing more work than necessary

## Setup: Creating Our Test Environment

In [ ]:
##This is just for local environments
# Start a SparkSession
#import findspark
#findspark.init()
#findspark.find()

'C:\\spark-3.5.7-bin-hadoop3'

In [ ]:
##AWS Glue config:
#This must be the very first cell you run in the notebook. If you run any other Spark code first, the session will start with default settings (AQE enabled), and you won't be able to change it without restarting the kernel.
#When AQE is enabled, Spark looks at the size of your data during execution. If your training dataset is small (like the Titanic CSV or a few thousand rows), AQE realizes that "n" partitions are unnecessary.
%%configure
{
    "--conf": "spark.sql.adaptive.enabled=false"
}

In [ ]:
# Cell 1: Setup PySpark Session
#from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import time
import pandas as pd

# Create Spark session with UI enabled
#spark = SparkSession.builder.appName("Performance Optimisation").getOrCreate()

# Set log level to reduce noise
#spark.sparkContext.setLogLevel("WARN")

print(f"Spark version: {spark.version}")
print(f"Spark UI available at: {spark.sparkContext.uiWebUrl}")
instances = spark.sparkContext.getConf().get("spark.executor.instances")
print(f"Requested Executors: {instances}")
slots = spark.sparkContext.defaultParallelism
print(f"Running with {slots} parallel threads.")

Spark version: 3.5.7
Spark UI available at: http://host.docker.internal:4041


In [ ]:
#Extracting datasets:
##Orders dataset: 
pdf = pd.read_csv("https://rockborne-bucket-01-cbs.s3.eu-west-2.amazonaws.com/DataSources/04_03_01_PySpark_PerformanceOpt/orders.csv")
orders_df = spark.createDataFrame(pdf)

##Customers dataset:
pdf = pd.read_csv("https://rockborne-bucket-01-cbs.s3.eu-west-2.amazonaws.com/DataSources/04_03_01_PySpark_PerformanceOpt/customers.csv")
customers_df = spark.createDataFrame(pdf)

##Products dataset:
pdf = pd.read_csv("https://rockborne-bucket-01-cbs.s3.eu-west-2.amazonaws.com/DataSources/04_03_01_PySpark_PerformanceOpt/products.csv")
products_df = spark.createDataFrame(pdf)

In [ ]:
print(f"Orders: {orders_df.count():,} records")
print(f"Products: {products_df.count():,} records")
print(f"Customers: {customers_df.count():,} records")

# Show sample data
print("\n--- Sample Orders ---")
orders_df.show(5)

print("--- Sample Products ---")
products_df.show(5)


print("--- Sample Customers ---")
customers_df.show(5)

## Understanding the Execution Plan

Before you can optimise, you need to understand what Spark is actually doing. The `explain()` method shows you the execution plan - think of it like a flight plan for your data.

### What explain() Does

The `explain()` method prints Spark's execution plan WITHOUT running the query. It shows:
- Where data is read from
- What transformations are applied
- Where data moves between executors (shuffles)
- The order of operations

### How to Read an Execution Plan

Plans are read **bottom to top**:
1. Bottom = data source (where data starts)
2. Middle = transformations (filters, joins, aggregations)
3. Top = final output

In [ ]:
# Cell 3: Basic explain() usage

revenue_by_region = orders_df.groupBy("region").agg(
                                                        F.sum(F.col("quantity") * F.col("unit_price")).alias("total_revenue")
                                                    )

print("=== Execution Plan ===")
revenue_by_region.explain()

=== Execution Plan ===
== Physical Plan ==
*(2) HashAggregate(keys=[region#6], functions=[sum((cast(quantity#3L as double) * unit_price#4))])
+- Exchange hashpartitioning(region#6, 200), ENSURE_REQUIREMENTS, [plan_id=98]
   +- *(1) HashAggregate(keys=[region#6], functions=[partial_sum((cast(quantity#3L as double) * unit_price#4))])
      +- *(1) Project [quantity#3L, unit_price#4, region#6]
         +- *(1) Scan ExistingRDD[order_id#0,customer_id#1L,product_id#2L,quantity#3L,unit_price#4,order_date#5,region#6,channel#7]




### What the operators mean (from bottom to top)

#### 1) `Scan ExistingRDD[...]`

Spark is reading the source dataset (here, an RDD backing your DataFrame).

* This creates the **initial partitions** (based on how the RDD/DataFrame was created).
* Spark will launch **one task per input partition** to scan it.

#### 2) `Project [quantity, unit_price, region]`

Spark keeps only the columns needed for the aggregation.

* This is a **narrow** step: no shuffle.
* Runs inside the same tasks as the scan.

#### 3) `HashAggregate ... partial_sum(...)`  (**partial aggregation**)

This is Spark doing a **map-side / local pre-aggregation**:

* Each task (working on one input partition) computes:

  * `value = cast(quantity as double) * unit_price`
  * then groups locally by `region` and produces **partial sums** per region.

Why this exists:

* It reduces data volume *before* the shuffle (very good for performance).

#### 4) `Exchange hashpartitioning(region, 200) ENSURE_REQUIREMENTS`  (**shuffle**)

This is the big moment.

* Spark needs **all rows for the same region** to end up together to compute the final sum correctly.
* So it performs a **shuffle** using **hashpartitioning(region, 200)**:

  * for each partial record, Spark computes a bucket: `hash(region) % 200`
  * sends it to that shuffle partition

**200** here is the number of **shuffle partitions** (very often controlled by `spark.sql.shuffle.partitions`).

What this implies operationally:

* The next stage will have **200 reduce-side tasks** (one per shuffle partition).
* Your cluster will run them in waves depending on total available cores.

#### 5) `HashAggregate ... sum(...)` (**final aggregation**)

After the shuffle, Spark performs the **final** aggregate:

* Each reduce task reads one shuffle partition (which contains many regions, not one per region)
* merges partial sums into the final `sum(...)` per region.

In [7]:
revenue_by_region.show()

+------+-------------------+
|region|      total_revenue|
+------+-------------------+
| South|3.477354010200002E8|
|  East|     3.4628243398E8|
|  West|3.464336209400002E8|
| North|3.480338258100001E8|
+------+-------------------+



**Sample output explained:**
```
== Physical Plan ==
HashAggregate(keys=[region], functions=[sum((quantity * unit_price))])
+- Exchange hashpartitioning(region, 200)          <- SHUFFLE! Data moves here
   +- HashAggregate(keys=[region], functions=[partial_sum(...)])
      +- Scan [region, quantity, unit_price]       <- Read data here
```


### Key Symbols to Watch For

**1. Exchange (SHUFFLE) - Most Important!**

A **shuffle** is when Spark redistributes data across executors over the network.

**Why shuffles are expensive:**
- Network I/O is 100-1000x slower than memory
- Data must be serialised, sent, and deserialised
- Creates disk I/O if data doesn't fit in memory
- All partitions must complete before next stage starts

**When shuffles occur:**
- `groupBy()` - data must be regrouped by key
- `join()` (non-broadcast) - matching rows must be co-located
- `repartition()` - explicitly redistributing data
- `distinct()` - duplicates must be found across all data
- `orderBy()` - data must be globally sorted

**2. BroadcastExchange - ✓ Efficient**

Small table copied to all executors. Much cheaper than shuffling.

**3. Sort** - Ordering data, can be expensive for global sorts.

**4. Filter** - Row selection, usually cheap (pushes down to data source).

**5. Project** - Column selection, efficient.


In [ ]:
# Cell 4: Identifying shuffles in plans

# Query WITH shuffle

# Query WITHOUT shuffle


Query WITH shuffle (groupBy):
== Physical Plan ==
*(2) HashAggregate(keys=[customer_id#1L], functions=[count(1)])
+- Exchange hashpartitioning(customer_id#1L, 200), ENSURE_REQUIREMENTS, [plan_id=127]
   +- *(1) HashAggregate(keys=[customer_id#1L], functions=[partial_count(1)])
      +- *(1) Project [customer_id#1L]
         +- *(1) Scan ExistingRDD[order_id#0,customer_id#1L,product_id#2L,quantity#3L,unit_price#4,order_date#5,region#6,channel#7]



--------------------------------------------------

Query WITHOUT shuffle (just filter):
== Physical Plan ==
*(1) Filter (isnotnull(quantity#3L) AND (quantity#3L > 5))
+- *(1) Scan ExistingRDD[order_id#0,customer_id#1L,product_id#2L,quantity#3L,unit_price#4,order_date#5,region#6,channel#7]




In [ ]:
# Cell 5: Exercise 3.1

# Query A
query_a = orders_df.filter(F.col("region") == "North")

# Query B  
query_b = orders_df.groupBy("region", "channel").agg(F.sum("quantity"))

# Query C
query_c = orders_df.groupBy("customer_id").count().groupBy("count").count()

# YOUR PREDICTIONS: A=___, B=___, C=___

print("Query A:"); query_a.explain()
print("\nQuery B:"); query_b.explain()
print("\nQuery C:"); query_c.explain()


Query A:
== Physical Plan ==
*(1) Filter (isnotnull(region#6) AND (region#6 = North))
+- *(1) Scan ExistingRDD[order_id#0,customer_id#1L,product_id#2L,quantity#3L,unit_price#4,order_date#5,region#6,channel#7]



Query B:
== Physical Plan ==
*(2) HashAggregate(keys=[region#6, channel#7], functions=[sum(quantity#3L)])
+- Exchange hashpartitioning(region#6, channel#7, 200), ENSURE_REQUIREMENTS, [plan_id=202]
   +- *(1) HashAggregate(keys=[region#6, channel#7], functions=[partial_sum(quantity#3L)])
      +- *(1) Project [quantity#3L, region#6, channel#7]
         +- *(1) Scan ExistingRDD[order_id#0,customer_id#1L,product_id#2L,quantity#3L,unit_price#4,order_date#5,region#6,channel#7]



Query C:
== Physical Plan ==
*(3) HashAggregate(keys=[count#116L], functions=[count(1)])
+- Exchange hashpartitioning(count#116L, 200), ENSURE_REQUIREMENTS, [plan_id=247]
   +- *(2) HashAggregate(keys=[count#116L], functions=[partial_count(1)])
      +- *(2) HashAggregate(keys=[customer_id#1L], functions=[c

'\nSOLUTIONS:\n- Query A: 0 shuffles (filter is a narrow transformation)\n- Query B: 1 shuffle (groupBy needs data regrouped)\n- Query C: 2 shuffles (first groupBy, then second groupBy on different key)\n'

## Partitioning Strategies

### Introduction

**What is partitioning?**

Partitioning is how Spark divides data into chunks for parallel processing. Each partition is processed by one task on one executor core.

**Why does partitioning matter?**

1. **Parallelism:** More partitions = more parallel tasks (up to cluster capacity)
2. **Memory:** Each partition must fit in executor memory
3. **Shuffle efficiency:** Right partitioning can eliminate shuffles
4. **Balance:** Uneven partitions cause some tasks to run much longer

**Default partitioning:**
- When reading data: Usually one partition per file
- After shuffles: Uses `spark.sql.shuffle.partitions` (default: 200)

### What is Data Skew?

**Data skew** = partitions with very different sizes. This is critical because:
- Spark jobs complete when ALL tasks finish
- One huge partition = one very slow task
- Other executors sit idle waiting

**Skew Ratio** = Largest Partition / Average Partition
- 1.0 = perfectly balanced
- \>2.0 = concerning
- \>5.0 = serious problem

In [10]:
##Funciton 1 fails sometimes: 
def analyse_partitions(df, name="DataFrame"):
    sizes = df.rdd.mapPartitions(lambda it: [sum(1 for _ in it)]).collect()
    avg_size = sum(sizes) / len(sizes)
    skew_ratio = max(sizes) / avg_size if avg_size > 0 else 0
    
    print(f"{name}: {len(sizes)} partitions, avg={avg_size:,.0f}, max={max(sizes):,}, skew={skew_ratio:.2f}x")
    return skew_ratio

from pyspark.sql import functions as F

def show_partition_sizes_optimized(df, name="DataFrame"):
    """Display the size of each partition using optimized Spark SQL"""
    
    # Add a column for partition ID and group by it
    partition_counts = (df.withColumn("partition_id", F.spark_partition_id())
                          .groupBy("partition_id")
                          .count()
                          .collect())
    
    # Extract sizes into a list
    partition_sizes = [row["count"] for row in partition_counts]
    num_partitions = df.rdd.getNumPartitions() # Metadata call, this is safe
    
    # Handle cases where some partitions might be empty and didn't show up in groupBy
    actual_count = len(partition_sizes)
    if actual_count < num_partitions:
        partition_sizes.extend([0] * (num_partitions - actual_count))

    total_rows = sum(partition_sizes)
    avg_size = total_rows / num_partitions
    max_size = max(partition_sizes)
    min_size = min(partition_sizes)
    skew = max_size / avg_size if avg_size > 0 else 0

    print(f"\n{name} Partition Analysis (Optimized):")
    print(f"  Total partitions: {num_partitions}")
    print(f"  Min partition size: {min_size:,}")
    print(f"  Max partition size: {max_size:,}")
    print(f"  Avg partition size: {avg_size:,.0f}")
    print(f"  Skew ratio (max/avg): {skew:.2f}x")

In [ ]:
##Evaluationg our dataset: 


Orders: 8 partitions, avg=125,000, max=125,504, skew=1.00x


Products_data: 8 partitions, avg=625, max=625, skew=1.00x


Customer: 8 partitions, avg=12,500, max=13,312, skew=1.06x


### repartition() vs coalesce()

| Aspect | repartition(n) | coalesce(n) |
|--------|----------------|-------------|
| Direction | Increase OR decrease | Decrease only |
| Shuffle | Yes (full redistribution) | No (combines existing) |
| Balance | Even partition sizes | May be uneven |
| Performance | Slower | Faster |
| Use when | Need even distribution | Reducing before write |


In [16]:
print(f"Original: {orders_df.rdd.getNumPartitions()} partitions")

repartitioned = orders_df.repartition(50)
print(f"After repartition(50): {repartitioned.rdd.getNumPartitions()}")
print("\nPlan shows Exchange (shuffle):")
repartitioned.explain()

##Up to this point the partition is defined but no executed.  

Original: 8 partitions
After repartition(50): 50

Plan shows Exchange (shuffle):
== Physical Plan ==
Exchange RoundRobinPartitioning(50), REPARTITION_BY_NUM, [plan_id=309]
+- *(1) Scan ExistingRDD[order_id#0,customer_id#1L,product_id#2L,quantity#3L,unit_price#4,order_date#5,region#6,channel#7]




In [ ]:
# Method 1: coalesce (no shuffle - fast but uneven)
start = time.time()

# Method 2: repartition (shuffle - slower but even)
start = time.time()


Coalesce time: 6.83s

Coalesced Partition Analysis (Optimized):
  Total partitions: 8
  Min partition size: 124,928
  Max partition size: 125,504
  Avg partition size: 125,000
  Skew ratio (max/avg): 1.00x

Repartition time: 8.90s

Repartitioned Partition Analysis (Optimized):
  Total partitions: 8
  Min partition size: 125,000
  Max partition size: 125,000
  Avg partition size: 125,000
  Skew ratio (max/avg): 1.00x


**Note:**
`.write.format("noop").mode("overwrite").save()` tells Spark to go through all the trouble of executing the join, shuffling the data, and processing the rows, but then discard the final result.

In [ ]:
##Compare performance: 
start = time.time()
orders_df.groupBy("customer_id").agg(
    F.count("order_id").alias("order_count"),
    F.sum(F.col("quantity") * F.col("unit_price")).alias("total_spend"),
    F.avg("unit_price").alias("avg_price")).count()
print(f"Exec time: {time.time() - start}s")





Exec time: 10.876121044158936s
Exec time: 12.168325185775757s
Exec time: 12.238742351531982s


In [ ]:
##Extracting Squewed dataset 'A' is 99% of the data: 
pdf = pd.read_csv("https://rockborne-bucket-01-cbs.s3.eu-west-2.amazonaws.com/DataSources/04_02_01_PySpark_Hybrid_Datasets/squewed_data.csv")
df = spark.createDataFrame(pdf)

In [ ]:
##Lets create a squewed dataset: 
from pyspark.sql import functions as F

# 2. Force Skew: Repartition by the 'key' column
# All "A"s must go to the same partition because they share the same key

print("Original Key-Based Skew:")
# You will see 1 or 2 huge partitions and many empty/tiny ones

# 3. The Coalesce Trap
print("\n--- After Coalesce(2) ---")
# Coalesce will just merge the empty partitions into the huge one.
# The skew ratio will likely stay very high.


# 4. The Repartition Cure
print("\n--- After Repartition(2) ---")
# Repartition (without a column name) ignores the keys and 
# shuffles data purely to balance the load.


Original Key-Based Skew:

DataFrame Partition Analysis (Optimized):
  Total partitions: 200
  Min partition size: 0
  Max partition size: 1,000,000
  Avg partition size: 5,000
  Skew ratio (max/avg): 199.98x

--- After Coalesce(2) ---

DataFrame Partition Analysis (Optimized):
  Total partitions: 2
  Min partition size: 100
  Max partition size: 1,000,000
  Avg partition size: 500,050
  Skew ratio (max/avg): 2.00x

--- After Repartition(2) ---

DataFrame Partition Analysis (Optimized):
  Total partitions: 2
  Min partition size: 500,050
  Max partition size: 500,050
  Avg partition size: 500,050
  Skew ratio (max/avg): 1.00x


In [ ]:
##Comparing performance:
start = time.time()
print(f"Original Exec time: {time.time() - start}s")

start = time.time()
print(f"Coalesce Exec time: {time.time() - start}s")

start = time.time()
print(f"Repartition Exec time: {time.time() - start}s")



Original Exec time: 7.4145495891571045s
Coalesce Exec time: 8.23918867111206s
Exec time: 9.033608198165894s


- Key-Based Example:
    - Action: df.repartition("city")
    - Behavior: All records for "New York" go to Partition 1. All records for "SmallTown" go to Partition 2.
    - Result: If New York has 10 million people and SmallTown has 500, Partition 1 is a "straggler" that slows down the whole job.

- Coalesce vs. Repartition Example:
    - Coalesce: It refuses to split that "New York" partition. It just looks for another partition to glue to it. You still have a massive bottleneck.

    - Repartition: It breaks the "Key" rule. It cuts the New York partition in half and moves half the data to another executor.


## Caching and Persistence

### Introduction

**What is caching?**

Caching tells Spark to keep a DataFrame in memory after computation, so it can be reused without recomputing. Without caching, Spark recomputes the entire lineage every time.

**When to cache:**
- DataFrame used multiple times
- DataFrame is expensive to compute
- DataFrame fits in memory

**When NOT to cache:**
- Only used once
- Too large for memory
- Source is already fast

In [ ]:
# Cell 9: Caching demonstration

# Without caching - recomputes each time
orders_filtered = orders_df.filter(F.col("region") == "North")

print("Without caching (two actions on same filtered data):")
start = time.time()
count1 = orders_filtered.count()
count2 = orders_filtered.groupBy("customer_id").count().count()
print(f"Time: {time.time() - start:.2f}s")


Without caching (two actions on same filtered data):
Time: 19.33s

With caching:
Time: 10.41s


DataFrame[order_id: string, customer_id: bigint, product_id: bigint, quantity: bigint, unit_price: double, order_date: string, region: string, channel: string]

### Persistence Levels

| Level | Storage | Use When |
|-------|---------|----------|
| MEMORY_ONLY | RAM | Default, fastest |
| MEMORY_AND_DISK | RAM + disk overflow | Data might not fit |
| MEMORY_ONLY_SER | RAM (compressed) | Memory tight |
| DISK_ONLY | Disk | Memory very limited |

In [26]:
##Testing persistence levels: 
import time
from pyspark import StorageLevel

# List of levels we want to test
levels = [
    ("NONE", None), # Baseline: No caching
    ("MEMORY_ONLY", StorageLevel.MEMORY_ONLY),
    ("MEMORY_AND_DISK", StorageLevel.MEMORY_AND_DISK),
    #("MEMORY_ONLY_SER", StorageLevel.MEMORY_ONLY_SER),
    ("DISK_ONLY", StorageLevel.DISK_ONLY)
]

print(f"{'Storage Level':<20} | {'Write Time (s)':<15} | {'Read Time (s)':<15}")
print("-" * 55)

for level_name, level_value in levels:
    # 1. Create the filtered DataFrame
    test_df = orders_df.filter(F.col("region") == "North")
    
    # 2. Apply persistence (except for NONE)
    if level_value:
        test_df.persist(level_value)
    
    # 3. MEASURE WRITE TIME (First Action)
    # This triggers the filter AND stores the result in the cache
    start_write = time.time()
    _ = test_df.count() 
    write_time = time.time() - start_write
    
    # 4. MEASURE READ TIME (Second Action)
    # This should be faster if the level is efficient
    start_read = time.time()
    _ = test_df.count()
    read_time = time.time() - start_read
    
    
    print(f"{level_name:<20} | {write_time:<15.2f} | {read_time:<15.2f}")
    
    # 5. CLEAN UP for the next iteration
    test_df.unpersist()

Storage Level        | Write Time (s)  | Read Time (s)  
-------------------------------------------------------
NONE                 | 6.78            | 6.73           
MEMORY_ONLY          | 6.65            | 0.21           
MEMORY_AND_DISK      | 6.78            | 0.11           
DISK_ONLY            | 7.07            | 0.11           


## Join Optimisation

### Introduction

Joins are common and expensive. For Spark to join two DataFrames:
1. Rows with matching keys must be on the SAME executor
2. This often requires shuffling BOTH DataFrames
3. Shuffling means network transfer, serialisation, and disk I/O

### Broadcast Joins

Instead of shuffling both tables, send the small table to every executor. Each executor then has a complete copy and can join locally.

**Benefits:**
- NO shuffle for the large table
- Small table sent once per executor
- Dramatically faster for large + small joins


In [27]:
from pyspark.sql.functions import broadcast

# Standard join (two shuffles)
print("Standard join plan:")
orders_df.join(products_df, "product_id").explain()

print("\n" + "-"*50 + "\n")

# Broadcast join (one broadcast, no shuffle on orders)
print("Broadcast join plan:")
orders_df.join(broadcast(products_df), "product_id").explain()

Standard join plan:
== Physical Plan ==
*(5) Project [product_id#2L, order_id#0, customer_id#1L, quantity#3L, unit_price#4, order_date#5, region#6, channel#7, product_name#17, category#18]
+- *(5) SortMergeJoin [product_id#2L], [product_id#16L], Inner
   :- *(2) Sort [product_id#2L ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(product_id#2L, 200), ENSURE_REQUIREMENTS, [plan_id=1986]
   :     +- *(1) Filter isnotnull(product_id#2L)
   :        +- *(1) Scan ExistingRDD[order_id#0,customer_id#1L,product_id#2L,quantity#3L,unit_price#4,order_date#5,region#6,channel#7]
   +- *(4) Sort [product_id#16L ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(product_id#16L, 200), ENSURE_REQUIREMENTS, [plan_id=1992]
         +- *(3) Filter isnotnull(product_id#16L)
            +- *(3) Scan ExistingRDD[product_id#16L,product_name#17,category#18]



--------------------------------------------------

Broadcast join plan:
== Physical Plan ==
*(2) Project [product_id#2L, order_

In [ ]:
# Performance comparison


--- Performance Comparison ---
Standard join: 18.90s
Broadcast join: 16.76s


### Broadcast Join Threshold

Spark automatically broadcasts tables below a threshold (default 10MB).



In [31]:
# Check current threshold
print(f"Auto broadcast threshold: {spark.conf.get('spark.sql.autoBroadcastJoinThreshold')}")

Auto broadcast threshold: 10485760b


## Addressing Data Skew via "Salting"

In distributed computing, a **Join** operation requires Spark to collocate rows with the same key on the same executor (a process called "shuffling"). **Data Skew** occurs when a single key has a disproportionately high number of records. This creates a "hot partition," where one executor performs the bulk of the work while others remain idle, leading to inefficient resource utilization and potential **OutOfMemory (OOM)** errors.

**Salting** is a sophisticated optimization technique used to artificially redistribute these "hot" keys across multiple partitions.

---

### The Example: E-commerce "Guest" Checkout

Consider a dataset of **Transactions** being joined with a **User_Profiles** table.

* **The Problem:** Many e-commerce platforms allow "Guest" checkouts. In the `Transactions` table, the `user_id` for these millions of rows might be a single default value: `0` or `NULL`.
* **The Bottleneck:** During a join, Spark will send *every* guest transaction to the exact same executor because they share the same `user_id`. That executor will likely crash or hang, even if the rest of the cluster is finished.

---

### The Implementation Strategy

#### 1. Salting the Fact Table (Transactions)

We append a random integer the "salt" to the skewed key. If we choose a salt range of 1 to 10, the original `user_id: 0` becomes ten distinct keys: `0_1, 0_2, ... 0_10`.

* **Result:** The guest transactions are now evenly distributed across 10 different executors.

#### 2. Exploding the Dimension Table (User_Profiles)

To ensure the join still works, we must modify the `User_Profiles` table to match these new keys. We take the single row for `user_id: 0` and **replicate** (explode) it 10 times, appending each possible salt value.

* **Result:** `user_id: 0` becomes 10 rows: `0_1, 0_2, ... 0_10`.

#### 3. The Salted Join

We perform the join on the new `salted_user_id` column. Because the data is now uniform, Spark can process the join in parallel across the entire cluster.

In [ ]:
# 1 million rows: 900,000 are Customer ID 1 (The Monster), 100,000 are others.
##Orders dataset: 
pdf = pd.read_csv("https://rockborne-bucket-01-cbs.s3.eu-west-2.amazonaws.com/DataSources/04_03_01_PySpark_PerformanceOpt/orders_skewed.csv")
orders_skewed_df = spark.createDataFrame(pdf)

##Customers dataset:
pdf = pd.read_csv("https://rockborne-bucket-01-cbs.s3.eu-west-2.amazonaws.com/DataSources/04_03_01_PySpark_PerformanceOpt/customers_skewed.csv")
customers_skewed_df = spark.createDataFrame(pdf)

In [61]:
# --- OPTION A: THE REGULAR JOIN (SKEWED) ---
print("Running Regular Join...")
start = time.time()
# Standard join on the skewed key
regular_join = orders_skewed_df.join(customers_skewed_df, "customer_id")
regular_join.count() 
print(f"Regular Join Finished in: {time.time() - start:.2f}s")

Running Regular Join...
Regular Join Finished in: 34.73s


In [62]:
# --- OPTION B: THE SALTED JOIN (BALANCED) ---
SALT_BUCKETS = 10

print(f"\nApplying Salt (Split into {SALT_BUCKETS} buckets)...")

# STEP 1: Salt the Large Table
# We turn ID '1' into '1_0', '1_1', '1_2', etc. randomly.
orders_salted = orders_skewed_df.withColumn("salt", (F.rand() * SALT_BUCKETS).cast("int")) \
                         .withColumn("salted_key", F.concat("customer_id", F.lit("_"), "salt"))

print("Example of Salted Orders (Customer 1 is now fragmented):")
orders_salted.select("customer_id", "salted_key").show(5)
orders_salted.count()



Applying Salt (Split into 10 buckets)...
Example of Salted Orders (Customer 1 is now fragmented):
+-----------+----------+
|customer_id|salted_key|
+-----------+----------+
|          1|       1_5|
|          1|       1_7|
|          1|       1_9|
|          1|       1_8|
|          1|       1_4|
+-----------+----------+
only showing top 5 rows



10100000

In [63]:
# STEP 2: Explode the Small Table
# We must replicate Customer 1 ten times so it can match any of the 10 possible salts.
customers_exploded = customers_skewed_df.withColumn("salt_array", F.array([F.lit(i) for i in range(SALT_BUCKETS)])) \
                                 .withColumn("salt", F.explode("salt_array")) \
                                 .withColumn("salted_key", F.concat("customer_id", F.lit("_"), "salt"))

print("Example of Exploded Customers (Customer 1 now has 10 entries):")
customers_exploded.select("customer_id", "salted_key").filter("customer_id == 1").show(5)

print(f"Original row count: {customers_skewed_df.count()}")
print(f"Exploded row count: {customers_exploded.count()}")


Example of Exploded Customers (Customer 1 now has 10 entries):
+-----------+----------+
|customer_id|salted_key|
+-----------+----------+
|          1|       1_0|
|          1|       1_1|
|          1|       1_2|
|          1|       1_3|
|          1|       1_4|
+-----------+----------+
only showing top 5 rows

Original row count: 100001
Exploded row count: 1000010


In [64]:

# STEP 3: The Salted Join
start = time.time()
salted_join = orders_salted.join(customers_exploded, "salted_key")
salted_join.count()
print(f"Salted Join Finished in: {time.time() - start:.2f}s")

Salted Join Finished in: 32.41s


**Is there a "Best Practice" for the number of buckets?**

There is no "magic number," but choosing the value is a balancing act between parallelism and memory overhead. Here are the professional guidelines for determining the value:

- Match your Cluster Resources: A common starting point is to align the salt buckets with the number of available executors or CPU cores.
- Consider the "Explosion" Cost: Remember that your small table (the `customers_df`) is multiplied by the `SALT_BUCKETS` value.
- The "Power of 2" Rule: Many engineers prefer powers of 2 (e.g., 16, 32, 64) because of how Spark handles internal hash partitioning, though this is less critical than the sheer volume of the split.


## Common Performance Anti-Patterns

### Introduction

Anti-patterns are common practices that seem reasonable but hurt performance.

### Anti-Pattern 1: Collecting Large Data

In [51]:
# Cell 20: Anti-Pattern 1

# BAD - pulls all 1M rows to driver
# all_data = orders_df.collect()  # DON'T!

# GOOD - use limit or aggregations

sample = orders_df.limit(100).collect()
#summary = orders_df.agg(F.count("*"), F.sum("quantity")).collect()
#summary
print(f"Safe: collected {len(sample)} samples")

Safe: collected 100 samples


### Anti-Pattern 2: Python UDFs

**What is a UDF?** A User Defined Function - custom Python code run on Spark data.

**Why UDFs are slow:** Data must be serialised to Python, processed slowly, serialised back. 10-100x slower than native functions.

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

# BAD - Python UDF
@udf(returnType=DoubleType())
def calc_total_udf(qty, price):
    return float(qty) * float(price)



UDF vs Native:
UDF:    6.78s
Native: 7.68s


### Anti-Pattern 3: No Caching for Reused Data


In [ ]:
# BAD - computes 3 times
expensive = orders_df.join(products_df, "product_id").groupBy("category").count()
# expensive.filter(...).count()
# expensive.orderBy(...).show()
# expensive.collect()

# GOOD - cache and reuse
expensive_cached = expensive.cache()
expensive_cached.count()  # Trigger
# expensive_cached.filter(...).count()
# expensive_cached.orderBy(...).show()
expensive_cached.unpersist()

### Anti-Pattern 4: Too Many or Too Few Partitions

Finding the right partition count

- Rule of thumb: 2-4 partitions per CPU core
- Or: ~128MB per partition for large data

In [ ]:
##CPU cores: 
#spark.sparkContext.defaultParallelism
#The total number of slots (cores) available across all executors. 
#In many clusters, this is Executors * Cores per Executor
spark.sparkContext.defaultParallelism

8

In [58]:
# Returns the configured number of instances
instances = spark.sparkContext.getConf().get("spark.executor.instances")
print(f"Requested Executors: {instances}")

Requested Executors: None


In [55]:
##This gives an approximation of the size in memory for each partition (in bytes).
##Note: sys.getsizeof measures Python object size, not Spark’s internal binary representation.

import sys
print("Number of partitions:", customers_df.rdd.getNumPartitions())
sizes_bytes = customers_df.rdd.mapPartitions(lambda rows: [sum(sys.getsizeof(row) for row in rows)]).collect()
sizes_mb = [round(size / (1024 * 1024), 2) for size in sizes_bytes]

print("Partition sizes (approx, in MB):", sizes_mb)


Number of partitions: 8
Partition sizes (approx, in MB): [0.84, 0.84, 0.84, 0.91, 0.84, 0.84, 0.84, 0.89]


In [ ]:
# Check your parallelism
default_parallelism = spark.sparkContext.defaultParallelism
print(f"Default parallelism: {default_parallelism}")

# Shuffle partitions (used after joins, aggregations)
shuffle_partitions = spark.conf.get("spark.sql.shuffle.partitions")
print(f"Shuffle partitions: {shuffle_partitions}")

# Adjust based on your data size
# For our 1M rows, 200 shuffle partitions is likely too many
spark.conf.set("spark.sql.shuffle.partitions", "200")

# Re-run aggregation with optimised partitions


Default parallelism: 8
Shuffle partitions: 200
== Physical Plan ==
*(2) HashAggregate(keys=[region#6], functions=[sum((cast(quantity#3L as double) * unit_price#4))])
+- Exchange hashpartitioning(region#6, 200), ENSURE_REQUIREMENTS, [plan_id=3479]
   +- *(1) HashAggregate(keys=[region#6], functions=[partial_sum((cast(quantity#3L as double) * unit_price#4))])
      +- *(1) Project [quantity#3L, unit_price#4, region#6]
         +- *(1) Scan ExistingRDD[order_id#0,customer_id#1L,product_id#2L,quantity#3L,unit_price#4,order_date#5,region#6,channel#7]




## Exercises

In [ ]:
#Exercise: Find and fix all anti-patterns:

from pyspark.sql.types import StringType
from pyspark.sql.functions import udf

@udf(returnType=StringType())
def size_udf(qty):
    return "Large" if qty >= 10 else "Small"

def bad_code():
    data = orders_df.join(products_df, "product_id")
    data = data.withColumn("size", size_udf("quantity"))
    r1 = data.groupBy("category").count().collect()
    r2 = data.groupBy("region").count().collect()
    return r1, r2

In [41]:
start = time.time()
bc1 = bad_code()
print(f"Bad code: {time.time() - start:.2f}s")

Bad code: 34.08s


In [ ]:
##Solution: 


Good code: 43.07s


In [44]:
## Exercise 2 - Three Independent Reports

def report_1():
    """Revenue by region"""
    return orders_df.join(products_df, "product_id") \
        .groupBy("region") \
        .agg(F.sum(F.col("quantity") * F.col("unit_price")).alias("revenue"))

def report_2():
    """Revenue by category"""
    return orders_df.join(products_df, "product_id") \
        .groupBy("category") \
        .agg(F.sum(F.col("quantity") * F.col("unit_price")).alias("revenue"))

def report_3():
    """Top 5 products by revenue"""
    return orders_df.join(products_df, "product_id") \
        .groupBy("product_id", "product_name") \
        .agg(F.sum(F.col("quantity") * F.col("unit_price")).alias("revenue")) \
        .orderBy(F.desc("revenue")) \
        .limit(5)

# Current approach - runs everything 3 times
start = time.time()
r1 = report_1().collect()
r2 = report_2().collect()
r3 = report_3().collect()
print(f"Three separate pipelines: {time.time() - start:.2f}s")

Three separate pipelines: 61.49s


In [45]:
#Exercise - YOUR SOLUTION

def optimised_reports():
    """
    EXERCISE: Generate all three reports efficiently
    
    Hints:
    1. What computation is shared across all three reports?
    2. Should you cache anything?
    3. Which join strategy should you use?
    
    Return a tuple of (report_1_df, report_2_df, report_3_df)
    """
    
    # YOUR CODE HERE
    pass

### Exercise 3: Fix a Skewed Aggregation

**Business Scenario:** Customer analytics is running slowly because a few enterprise customers have millions of transactions while most have only a few.

In [ ]:
## Exercise: Fix a Skewed Aggregation
###Customer analytics is running slowly because a few enterprise customers have millions of transactions while most have only a few.
# Create extremely skewed data

skewed_data = []
for i in range(500_000):
    # 3 enterprise customers with 100K transactions each
    if i < 300_000:
        customer = i % 3 + 1  # Customers 1, 2, 3
    else:
        customer = random.randint(4, 50000)  # Everyone else
    
    skewed_data.append((
        f"TXN-{i}",
        customer,
        random.uniform(10, 1000),
        random.choice(['2024-01-15', '2024-01-16', '2024-01-17'])
    ))

skewed_df = spark.createDataFrame(
    skewed_data,
    ["transaction_id", "customer_id", "amount", "date"]
)

# Show the skew
print("Transaction distribution:")
skewed_df.groupBy("customer_id").count() \
    .orderBy(F.desc("count")).show(10)

Transaction distribution:
+-----------+------+
|customer_id| count|
+-----------+------+
|          1|100000|
|          2|100000|
|          3|100000|
|      48343|    14|
|       1520|    14|
|      40384|    14|
|      21815|    14|
|       7584|    13|
|       2465|    13|
|      45120|    13|
+-----------+------+
only showing top 10 rows



In [ ]:
# Exercise 3 - YOUR SOLUTION

def handle_skewed_aggregation(df):
    """
    EXERCISE: Calculate total spend per customer efficiently
    despite the severe skew in customer_id
    
    Hints:
    1. Can you identify and handle heavy keys differently?
    2. Consider salting
    3. Consider two-phase aggregation 
    
    Return: DataFrame with customer_id, total_spend
    """
    
    # YOUR CODE HERE
    pass

## Key Takeaways

### Performance Checklist

Before running any PySpark job, ask yourself:

1. **Have I checked the execution plan?**
   - Use `explain()` to understand what Spark will actually do
   - Count the shuffles (Exchange operations)

2. **Are my partitions sized correctly?**
   - Target 128MB per partition for large data
   - 2-4 partitions per CPU core
   - Use `coalesce()` to reduce, `repartition()` to increase or balance

3. **Have I used broadcast joins where appropriate?**
   - Any table under 100MB is a candidate
   - Use `broadcast()` hint or adjust threshold

4. **Am I reusing computed data efficiently?**
   - Cache DataFrames used multiple times
   - Always `unpersist()` when done

5. **Am I using native functions instead of UDFs?**
   - Native functions run in JVM (fast)
   - Python UDFs require serialisation (slow)

6. **Is my data skewed?**
   - Check partition sizes
   - Use salting for severely skewed keys

### Quick Reference Commands

```python
# Check partitions
df.rdd.getNumPartitions()

# View execution plan
df.explain(mode="formatted")

# Change shuffle partitions
spark.conf.set("spark.sql.shuffle.partitions", "100")

# Broadcast small table
from pyspark.sql.functions import broadcast
df1.join(broadcast(df2), "key")

# Cache and unpersist
df.cache()
df.unpersist()

# Check cached data
spark.catalog.clearCache()
```


## Further Reading

To deepen your understanding of PySpark performance optimisation techniques, consider the following resources:

- **Spark SQL Performance Tuning Guide:** Official Apache Spark guide covering caching, Adaptive Query Execution (AQE), join hints, partition coalescing, and skew handling. Essential reading for optimising Spark SQL workloads. [https://spark.apache.org/docs/latest/sql-performance-tuning.html](https://spark.apache.org/docs/latest/sql-performance-tuning.html)

- **Spark Tuning Guide:** Core Spark tuning documentation covering data serialization, memory management, garbage collection tuning, and data locality. Foundational for understanding Spark internals and resource optimisation. [https://spark.apache.org/docs/latest/tuning.html](https://spark.apache.org/docs/latest/tuning.html)

- **Databricks Performance Best Practices:** Enterprise-focused guide covering horizontal scaling, partitioning strategies, Z-ordering, statistics collection, and query optimization for production workloads. [https://docs.databricks.com/en/lakehouse-architecture/performance-efficiency/best-practices](https://docs.databricks.com/en/lakehouse-architecture/performance-efficiency/best-practices)

- **AWS Spark Performance Tuning Strategies:** Comprehensive guide covering performance goals, metrics identification, bottleneck analysis, and optimization strategies. Applicable beyond AWS Glue to general Spark tuning workflows. [https://docs.aws.amazon.com/prescriptive-guidance/latest/tuning-aws-glue-for-apache-spark/performance-tuning-strategies.html](https://docs.aws.amazon.com/prescriptive-guidance/latest/tuning-aws-glue-for-apache-spark/performance-tuning-strategies.html)